In [ ]:
!pip install -q \
transformers \
datasets \
peft \
accelerate \
bitsandbytes \
sentencepiece \
huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!find "/content/drive/MyDrive" -name "*.ipynb"

/content/drive/MyDrive/Colab Notebooks/02_qwen_evaluation.ipynb
/content/drive/MyDrive/Colab Notebooks/05_memory_rag.ipynb
/content/drive/MyDrive/Colab Notebooks/07_XAI_Finaldemo.ipynb
/content/drive/MyDrive/Colab Notebooks/06_qwen_inference.ipynb
/content/drive/MyDrive/Colab Notebooks/02_qwen_finetuning_03_therapy_rag.ipynb


In [1]:
!git clone https://github.com/Nityakothavari7/EmberMind.git

Cloning into 'EmberMind'...
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (3/3), done.


In [2]:
%cd EmberMind

/content/EmberMind


In [4]:
!git config --global user.name "Nityakothavari7"
!git config --global user.email "nityakothavari@gmail.com"

In [5]:
!cp "/content/02_qwen_finetuning_03_therapy_rag.ipynb" "/content/EmberMind/"

cp: cannot stat '/content/02_qwen_finetuning_03_therapy_rag.ipynb': No such file or directory


In [ ]:
import json
import pandas as pd
from datasets import load_dataset

mentalchat = load_dataset("ShenLab/MentalChat16K")
esconv = load_dataset("thu-coai/esconv")

records = []

# --------------------
# MentalChat16K
# --------------------

for row in mentalchat["train"]:

    records.append({
        "instruction": row["instruction"],
        "input": row["input"],
        "output": row["output"]
    })

# --------------------
# ESConv
# --------------------

strategy_map = {

    "Reflection of feelings":
    "<STRATEGY_REFLECTION>",

    "Affirmation and Reassurance":
    "<STRATEGY_AFFIRMATION>",

    "Question":
    "<STRATEGY_QUESTION>",

    "Providing Suggestions":
    "<STRATEGY_SUGGESTION>",

    "Restatement or Paraphrasing":
    "<STRATEGY_PARAPHRASE>",

    "Information":
    "<STRATEGY_INFORMATION>"
}

for sample in esconv["train"]:

    try:

        conversation = json.loads(
            sample["text"]
        )

        dialog = conversation["dialog"]

        user_msgs = []
        final_response = None
        strategy_token = "<STRATEGY_GENERAL>"

        for turn in dialog:

            if turn["speaker"] == "usr":

                user_msgs.append(
                    turn["text"]
                )

            elif turn["speaker"] == "sys":

                final_response = turn["text"]

                if "strategy" in turn:

                    strategy_token = strategy_map.get(
                        turn["strategy"],
                        "<STRATEGY_GENERAL>"
                    )

        if (
            len(user_msgs) > 0
            and final_response is not None
        ):

            records.append({

                "instruction":
                strategy_token,

                "input":
                "\n".join(user_msgs),

                "output":
                final_response
            })

    except:
        pass

df = pd.DataFrame(records)

print("Total Samples:", len(df))

print(df.tail())

df.to_json(
    "ember_train_final.json",
    orient="records",
    indent=2
)

print("Saved!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.95k [00:00<?, ?B/s]

Interview_Data_6K.csv:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Synthetic_Data_10K.csv:   0%|          | 0.00/32.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16084 [00:00<?, ? examples/s]

...

test.txt:   0%|          | 0.00/881k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/910 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/195 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/195 [00:00<?, ? examples/s]

Total Samples: 16994
                  instruction  \
16989      <STRATEGY_GENERAL>   
16990   <STRATEGY_SUGGESTION>   
16991      <STRATEGY_GENERAL>   
16992  <STRATEGY_AFFIRMATION>   
16993      <STRATEGY_GENERAL>   

                                                   input  \
16989  hi\nI'm in a big mess\nI am not fine , I tried...   
16990  Hi, how are you doing today?\nYes I am. How ar...   
16991  hello\nso am having a hard time with my friend...   
16992  hello\nstressed out, anxious , this covid life...   
16993  I'm so sad\nMy partner left me for another wom...   

                                                  output  
16989                                          good luck  
16990  The study suggests that the story be upbeat an...  
16991                                       Same to you!  
16992  I understand that feeling! well hopefully we g...  
16993                   I'm so happy I was able to help.  
Saved!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp ember_train_final.json /content/drive/MyDrive/

In [ ]:
!ls /content/drive/MyDrive | grep ember

ember_qwen_checkpoints
ember_qwen_lora
ember_qwen_lora.zip
ember_train_final.json


In [ ]:
import torch
import pandas as pd

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from google.colab import drive
drive.mount('/content/drive')
# =====================================================
# LOAD DATASET
# =====================================================

df = pd.read_json("ember_train_final.json")

print("Total Samples:", len(df))

dataset = Dataset.from_pandas(df)

# =====================================================
# FORMAT DATA
# =====================================================

def format_example(example):

    text = f"""### Instruction:
{example['instruction']}

### User:
{example['input']}

### Therapist:
{example['output']}"""

    return {"text": text}

dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

dataset = dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

# =====================================================
# LOAD GEMMA
# =====================================================

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

# =====================================================
# PREPARE FOR QLORA
# =====================================================

model = prepare_model_for_kbit_training(model)

# =====================================================
# TOKENIZATION
# =====================================================

def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

train_dataset = train_dataset.map(tokenize)
eval_dataset = eval_dataset.map(tokenize)

# remove raw text

train_dataset = train_dataset.remove_columns(["text"])
eval_dataset = eval_dataset.remove_columns(["text"])

# =====================================================
# LORA
# =====================================================

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],

    lora_dropout=0.05,

    bias="none",

    task_type="CAUSAL_LM",
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

# =====================================================
# DATA COLLATOR
# =====================================================

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# =====================================================
# TRAINING ARGUMENTS
# =====================================================

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ember_qwen_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=25,

    eval_strategy="steps",
    eval_steps=250,

    save_strategy="steps",
    save_steps=250,

    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    report_to="none",
)

# =====================================================
# TRAINER
# =====================================================

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    data_collator=data_collator,
)

# =====================================================
# TRAIN
# =====================================================

print("\nStarting Training...\n")

trainer.train()

# =====================================================
# SAVE MODEL
# =====================================================

FINAL_MODEL_DIR = "/content/drive/MyDrive/ember_qwen_lora"

model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("\nModel Saved!")

# =====================================================
# ZIP MODEL
# =====================================================

!zip -r /content/drive/MyDrive/ember_qwen_lora.zip \
    /content/drive/MyDrive/ember_qwen_lora

print("\nZIP CREATED!")

# =====================================================
# FINAL STATS
# =====================================================

print(trainer.state)

Mounted at /content/drive
Total Samples: 16994


Map:   0%|          | 0/16994 [00:00<?, ? examples/s]

Train: 16144
Eval : 850


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

Step,Training Loss,Validation Loss
250,0.992857,1.012732
500,1.000319,0.982739
750,0.981250,0.968578


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

Step,Training Loss,Validation Loss
250,0.992857,1.012732
500,1.000319,0.982739
750,0.981250,0.968578
1000,0.960667,0.962046
1009,0.960667,0.962039


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



Model Saved!
  adding: content/drive/MyDrive/ember_qwen_lora/ (stored 0%)
  adding: content/drive/MyDrive/ember_qwen_lora/README.md (deflated 65%)
  adding: content/drive/MyDrive/ember_qwen_lora/adapter_model.safetensors (deflated 8%)
  adding: content/drive/MyDrive/ember_qwen_lora/adapter_config.json (deflated 59%)
  adding: content/drive/MyDrive/ember_qwen_lora/chat_template.jinja (deflated 71%)
  adding: content/drive/MyDrive/ember_qwen_lora/tokenizer_config.json (deflated 60%)
  adding: content/drive/MyDrive/ember_qwen_lora/tokenizer.json (deflated 81%)

ZIP CREATED!
TrainerState(epoch=1.0, global_step=1009, max_steps=1009, logging_steps=25, eval_steps=250, save_steps=250, train_batch_size=2, num_train_epochs=1, num_input_tokens_seen=0, total_flos=1.3797880032657408e+17, log_history=[{'loss': 1.3168049621582032, 'grad_norm': 0.24512827396392822, 'learning_rate': 0.0001952428146679881, 'epoch': 0.024777006937561942, 'step': 25}, {'loss': 1.1664618682861327, 'grad_norm': 0.242195069

In [ ]:
!du -sh /content/drive/MyDrive/ember_qwen_lora
!ls -lh /content/drive/MyDrive/ember_qwen_lora.zip

du: cannot access '/content/drive/MyDrive/ember_qwen_lora': No such file or directory
ls: cannot access '/content/drive/MyDrive/ember_qwen_lora.zip': No such file or directory


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!find /content/drive/MyDrive -name "*qwen*" 2>/dev/null

Mounted at /content/drive
/content/drive/MyDrive/ember_qwen_checkpoints
/content/drive/MyDrive/ember_qwen_lora
/content/drive/MyDrive/ember_qwen_lora.zip


In [ ]:
!ls -lh /content/drive/MyDrive/ember_qwen_lora.zip

-rw------- 1 root root 29M Jun 10 11:14 /content/drive/MyDrive/ember_qwen_lora.zip


In [ ]:
!zip -r ember_qwen_checkpoints.zip /content/drive/MyDrive/ember_qwen_checkpoints

  adding: content/drive/MyDrive/ember_qwen_checkpoints/ (stored 0%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/ (stored 0%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/README.md (deflated 65%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/adapter_model.safetensors (deflated 8%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/adapter_config.json (deflated 59%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/chat_template.jinja (deflated 71%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/tokenizer_config.json (deflated 60%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/tokenizer.json (deflated 81%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/training_args.bin (deflated 54%)
  adding: content/drive/MyDrive/ember_qwen_checkpoints/checkpoint-1009/optimizer.pt (deflated 8%)
  adding: content/

In [ ]:
!pip install -q faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 30.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd

df = pd.read_json("ember_train_final.json")

print("Total Samples:", len(df))
print()

print(df.head())

Total Samples: 16994

                                         instruction  \
0  You are a helpful mental health counselling as...   
1  You are a helpful mental health counselling as...   
2  You are a helpful mental health counselling as...   
3  You are a helpful mental health counselling as...   
4  You are a helpful mental health counselling as...   

                                               input  \
0  I've been struggling with my mental health for...   
1  I've been feeling overwhelmed with my caregivi...   
2  I've been feeling constantly anxious and unabl...   
3  My mom has Alzheimer's, and I've been her prim...   
4  I've tried setting boundaries, but it feels li...   

                                              output  
0  I understand that you've been dealing with a s...  
1  Your situation is complex, and it's important ...  
2  I can see that you're dealing with a great dea...  
3  I'm sorry to hear that your siblings' demands ...  
4  Your concerns are valid, a

In [ ]:
documents = []

for _, row in df.iterrows():

    doc = f"""
User:
{row['input']}

Therapist:
{row['output']}
"""

    documents.append(doc)

print("Documents Created:", len(documents))
print()
print(documents[0][:500])

Documents Created: 16994


User:
I've been struggling with my mental health for a while now, and I can't seem to find a way to cope with it. I've tried visualization, positive thinking, and even medication, but nothing seems to work. I've been feeling lost and helpless, and I don't know what to do next. My mind is a whirlwind of thoughts and emotions, and I can't seem to make sense of it all. I feel like I'm drowning in a sea of confusion, and I can't seem to find my way out.

Therapist:
I understand that you've been dea


In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded


In [ ]:
embeddings = embedding_model.encode(
    documents,
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)

print(embeddings.shape)

Batches:   0%|          | 0/266 [00:00<?, ?it/s]

(16994, 384)


In [ ]:
import numpy as np

np.save(
    "/content/drive/MyDrive/therapy_embeddings.npy",
    embeddings
)

print("Embeddings Saved")

Embeddings Saved


In [ ]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

therapy_index = faiss.IndexFlatL2(dimension)

therapy_index.add(
    embeddings.astype(np.float32)
)

print("Total Vectors:", therapy_index.ntotal)

Total Vectors: 16994


In [ ]:
faiss.write_index(
    therapy_index,
    "/content/drive/MyDrive/therapy_index.faiss"
)

print("FAISS Index Saved")

FAISS Index Saved


In [ ]:
import pickle

with open(
    "/content/drive/MyDrive/therapy_metadata.pkl",
    "wb"
) as f:

    pickle.dump(
        documents,
        f
    )

print("Metadata Saved")

Metadata Saved


In [ ]:
query = """
I feel lonely and exhausted.
Nobody understands me.
"""

query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True
)

distances, indices = therapy_index.search(
    query_embedding.astype(np.float32),
    k=3
)

print("Retrieved Indices:")
print(indices)

Retrieved Indices:
[[ 6983 11814  9146]]


In [ ]:
for idx in indices[0]:

    print("="*80)

    print(documents[idx][:1500])

    print("\n")


User:
I've been feeling really lonely lately, even though I have friends and acquaintances around me. It's like there's a constant void that I can't fill. I crave deeper connections and meaningful relationships, but I struggle to open up and trust others. It feels like I'm always on the outside looking in, and it's starting to take a toll on my mental health. I want to learn how to build healthier relationships and overcome this sense of loneliness.

Therapist:
I can understand how difficult it must be to feel lonely even when you have people around you. It sounds like you're longing for more meaningful connections and struggling with trust issues. Building healthier relationships takes time and effort, but there are steps you can take to overcome this sense of loneliness.

Firstly, try to focus on self-care and nurturing your own well-being. Take some time each day to engage in activities that bring you joy and fulfillment. This could be pursuing hobbies, practicing mindfulness or se